# GeoNode and GeoServer Integration Tests (Improved)

This notebook provides comprehensive testing for GeoNode-GeoServer integration with improved security, error handling, and code quality.

## Features
- Secure credential management using environment variables
- Robust error handling and retry mechanisms
- Comprehensive input validation
- Improved logging and monitoring
- Type hints and documentation
- Modular and maintainable code structure

## Parameters
This cell is tagged with `parameters`. Papermill will inject values here when the notebook is executed via Kestra.

In [ ]:
# Configuration parameters (will be overridden by Papermill)
# Security Note: In production, use environment variables or secure credential management
import os
from typing import Dict, Any, Optional, List, Tuple

# Service URLs
GEONODE_URL = os.getenv("GEONODE_URL", "http://localhost:8000")
GEOSERVER_URL = os.getenv("GEOSERVER_URL", "http://localhost:8080/geoserver")

# Authentication (use environment variables in production)
USERNAME = os.getenv("GEONODE_USERNAME", "admin")
PASSWORD = os.getenv("GEONODE_PASSWORD", "geoserver")

# Test data paths
SAMPLE_SHAPEFILE_PATH = os.getenv("SAMPLE_SHAPEFILE_PATH", "../data/sample_vector.shp")
SAMPLE_RASTER_PATH = os.getenv("SAMPLE_RASTER_PATH", "../data/sample_raster.tif")

# Configuration constants
DEFAULT_TIMEOUT = 30  # seconds
UPLOAD_TIMEOUT = 180  # seconds for file uploads
MAX_RETRY_ATTEMPTS = 3
DEFAULT_WORKSPACE = "geonode"
MAX_FILE_SIZE_MB = 100  # Maximum file size for uploads

# Test configuration
ENABLE_CLEANUP = os.getenv("ENABLE_CLEANUP", "false").lower() == "true"
VERBOSE_LOGGING = os.getenv("VERBOSE_LOGGING", "false").lower() == "true"

# Variables to store test results
test_results: Dict[str, Dict[str, Any]] = {}
uploaded_layers: List[str] = []  # Track uploaded layers for cleanup

## Library Imports and Configuration

In [ ]:
import requests
import json
import logging
import time
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.parse import urljoin, urlparse
from requests.auth import HTTPBasicAuth
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import warnings
from contextlib import contextmanager
from datetime import datetime

# Optional imports with fallbacks
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    warnings.warn("GeoPandas not available. Some advanced features may be limited.")

try:
    from owslib.wms import WebMapService
    from owslib.wfs import WebFeatureService
    HAS_OWSLIB = True
except ImportError:
    HAS_OWSLIB = False
    warnings.warn("OWSLib not available. Some OGC service tests may be limited.")

# Configure logging with better formatting
log_level = logging.DEBUG if VERBOSE_LOGGING else logging.INFO
logging.basicConfig(
    level=log_level,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Suppress urllib3 warnings for unverified HTTPS requests
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

logger.info(f"Starting GeoNode-GeoServer integration tests at {datetime.now()}")
logger.info(f"GeoNode URL: {GEONODE_URL}")
logger.info(f"GeoServer URL: {GEOSERVER_URL}")
logger.info(f"GeoPandas available: {HAS_GEOPANDAS}")
logger.info(f"OWSLib available: {HAS_OWSLIB}")

## Utility Functions
Core utility functions for session management, validation, and error handling.

In [ ]:
# --- Utility Functions ---

class TestResult:
    """Class to represent test results with metadata."""
    def __init__(self, name: str, success: bool, message: str = "", duration: float = 0.0, metadata: Dict = None):
        self.name = name
        self.success = success
        self.message = message
        self.duration = duration
        self.timestamp = datetime.now().isoformat()
        self.metadata = metadata or {}
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "status": "SUCCESS" if self.success else "FAILURE",
            "message": self.message,
            "duration": self.duration,
            "timestamp": self.timestamp,
            "metadata": self.metadata
        }

@contextmanager
def timer():
    """Context manager to measure execution time."""
    start = time.time()
    yield
    end = time.time()
    return end - start

def record_test_result(test_name: str, success: bool, message: str = "", duration: float = 0.0, metadata: Dict = None) -> None:
    """
    Records a test result with enhanced metadata.
    
    Args:
        test_name: Name of the test
        success: Whether the test passed
        message: Descriptive message about the test result
        duration: Time taken to execute the test
        metadata: Additional metadata about the test
    """
    result = TestResult(test_name, success, message, duration, metadata)
    test_results[test_name] = result.to_dict()
    
    if success:
        logger.info(f"✓ Test '{test_name}': SUCCESS. {message}")
    else:
        logger.error(f"✗ Test '{test_name}': FAILURE. {message}")

def validate_url(url: str) -> bool:
    """
    Validates if a URL is properly formatted and uses allowed schemes.
    
    Args:
        url: URL to validate
        
    Returns:
        True if URL is valid, False otherwise
    """
    try:
        parsed = urlparse(url)
        return parsed.scheme in ['http', 'https'] and bool(parsed.netloc)
    except Exception:
        return False

def validate_file_path(file_path: str, max_size_mb: int = MAX_FILE_SIZE_MB) -> Tuple[bool, str]:
    """
    Validates if a file path exists and meets size requirements.
    
    Args:
        file_path: Path to the file
        max_size_mb: Maximum allowed file size in MB
        
    Returns:
        Tuple of (is_valid, error_message)
    """
    try:
        path = Path(file_path)
        if not path.exists():
            return False, f"File does not exist: {file_path}"
        
        if not path.is_file():
            return False, f"Path is not a file: {file_path}"
        
        size_mb = path.stat().st_size / (1024 * 1024)
        if size_mb > max_size_mb:
            return False, f"File too large: {size_mb:.2f}MB > {max_size_mb}MB"
        
        return True, ""
    except Exception as e:
        return False, f"Error validating file: {e}"

def sanitize_layer_name(name: str) -> str:
    """
    Sanitizes a layer name for use in URLs and identifiers.
    
    Args:
        name: Original layer name
        
    Returns:
        Sanitized layer name
    """
    # Remove or replace problematic characters
    sanitized = re.sub(r'[^a-zA-Z0-9_-]', '_', name)
    # Ensure it doesn't start with a number
    if sanitized and sanitized[0].isdigit():
        sanitized = f"layer_{sanitized}"
    return sanitized or "unnamed_layer"

## Session Management and Authentication
Secure session handling with retry mechanisms and proper error handling.

In [ ]:
# --- Session Management and Authentication ---

def create_robust_session(timeout: int = DEFAULT_TIMEOUT) -> requests.Session:
    """
    Creates a requests session with retry strategy and proper timeout handling.
    
    Args:
        timeout: Default timeout for requests
        
    Returns:
        Configured requests session
    """
    session = requests.Session()
    
    # Configure retry strategy
    retry_strategy = Retry(
        total=MAX_RETRY_ATTEMPTS,
        status_forcelist=[429, 500, 502, 503, 504],
        method_whitelist=["HEAD", "GET", "OPTIONS"],
        backoff_factor=1
    )
    
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    
    # Set default headers
    session.headers.update({
        'User-Agent': 'GeoNode-GeoServer-Test-Suite/1.0',
        'Accept': 'application/json, text/html, */*',
        'Accept-Language': 'en-US,en;q=0.9'
    })
    
    return session

def extract_csrf_token_safely(html_content: str) -> Optional[str]:
    """
    Safely extracts CSRF token from HTML content using multiple methods.
    
    Args:
        html_content: HTML content to parse
        
    Returns:
        CSRF token if found, None otherwise
    """
    if not html_content or len(html_content.strip()) == 0:
        return None
        
    # Method 1: Look for csrfmiddlewaretoken in forms (more secure patterns)
    patterns = [
        r'name=["\']csrfmiddlewaretoken["\']\s+value=["\']([a-zA-Z0-9]{32,})["\']',
        r'value=["\']([a-zA-Z0-9]{32,})["\']\s+name=["\']csrfmiddlewaretoken["\']',
        r'<input[^>]*name=["\']csrfmiddlewaretoken["\'][^>]*value=["\']([a-zA-Z0-9]{32,})["\']',
    ]
    
    for pattern in patterns:
        try:
            match = re.search(pattern, html_content, re.IGNORECASE | re.DOTALL)
            if match:
                token = match.group(1).strip()
                # Basic validation: CSRF tokens should be alphanumeric and reasonably long
                if token and len(token) >= 32 and token.replace('-', '').replace('_', '').isalnum():
                    return token
        except re.error as e:
            logger.warning(f"Regex error in CSRF token extraction: {e}")
            continue
    
    return None

def get_geonode_session(geonode_url: str, username: str, password: str) -> Optional[requests.Session]:
    """
    Establishes an authenticated session with GeoNode using secure methods.
    
    Args:
        geonode_url: Base URL of GeoNode instance
        username: Username for authentication
        password: Password for authentication
        
    Returns:
        Authenticated session or None if authentication fails
    """
    start_time = time.time()
    
    # Input validation
    if not all([geonode_url, username, password]):
        logger.error("Missing required authentication parameters")
        record_test_result("geonode_session_validation", False, "Missing authentication parameters")
        return None
        
    if not validate_url(geonode_url):
        logger.error(f"Invalid GeoNode URL: {geonode_url}")
        record_test_result("geonode_session_validation", False, f"Invalid URL: {geonode_url}")
        return None
        
    session = create_robust_session()
    login_url = urljoin(geonode_url.rstrip('/') + '/', 'account/login/')
    
    try:
        # First, get the login page to retrieve CSRF token
        logger.info(f"Accessing GeoNode login page: {login_url}")
        response = session.get(login_url, timeout=DEFAULT_TIMEOUT)
        response.raise_for_status()
        
        # Try to get CSRF token from cookies first
        csrf_token = session.cookies.get('csrftoken')
        
        # If not in cookies, extract from HTML content
        if not csrf_token:
            csrf_token = extract_csrf_token_safely(response.text)
            
        if not csrf_token:
            logger.error("Could not retrieve CSRF token from GeoNode login page")
            record_test_result("geonode_csrf_token", False, "CSRF token not found")
            return None
            
        logger.info("Successfully retrieved CSRF token")
        record_test_result("geonode_csrf_token", True, "CSRF token retrieved successfully")
        
    except requests.exceptions.RequestException as e:
        logger.error(f"Error accessing GeoNode login page: {e}")
        record_test_result("geonode_login_page_access", False, f"Request error: {e}")
        return None
    
    # Prepare login data
    login_data = {
        'username': username,
        'password': password,
        'csrfmiddlewaretoken': csrf_token,
        'next': '/account/profile/'  # Redirect to profile page after login
    }
    
    headers = {
        'Referer': login_url,
        'X-CSRFToken': csrf_token
    }
    
    try:
        # Attempt login
        logger.info("Attempting to log into GeoNode")
        response = session.post(login_url, data=login_data, headers=headers, timeout=DEFAULT_TIMEOUT)
        response.raise_for_status()
        
        # Check if login was successful
        if response.url.endswith('/account/login/'):
            # Check for specific error messages
            if "Please enter a correct username and password" in response.text:
                error_msg = "Invalid credentials"
            elif "This account is inactive" in response.text:
                error_msg = "Account is inactive"
            else:
                error_msg = f"Login failed - redirected back to login page"
            
            logger.error(f"GeoNode login failed: {error_msg}")
            record_test_result("geonode_login", False, error_msg, time.time() - start_time)
            return None
        
        # Login successful
        duration = time.time() - start_time
        logger.info(f"Successfully logged into GeoNode: {geonode_url}")
        record_test_result("geonode_login", True, "Login successful", duration)
        return session
        
    except requests.exceptions.RequestException as e:
        logger.error(f"GeoNode login request failed: {e}")
        record_test_result("geonode_login", False, f"Request error: {e}", time.time() - start_time)
        return None

def get_geoserver_auth() -> HTTPBasicAuth:
    """
    Returns the authentication object for GeoServer requests.
    
    Returns:
        HTTPBasicAuth object for GeoServer authentication
    """
    return HTTPBasicAuth(USERNAME, PASSWORD)

def check_geoserver_status(geoserver_url: str) -> bool:
    """
    Checks if GeoServer is running and accessible.
    
    Args:
        geoserver_url: Base URL of GeoServer instance
        
    Returns:
        True if GeoServer is accessible, False otherwise
    """
    start_time = time.time()
    
    if not validate_url(geoserver_url):
        logger.error(f"Invalid GeoServer URL: {geoserver_url}")
        record_test_result("geoserver_status_check", False, f"Invalid URL: {geoserver_url}")
        return False
    
    try:
        session = create_robust_session()
        url = urljoin(geoserver_url.rstrip('/') + '/', 'rest/workspaces.json')
        
        logger.info(f"Checking GeoServer status at: {url}")
        response = session.get(url, auth=get_geoserver_auth(), timeout=DEFAULT_TIMEOUT)
        
        if response.status_code == 200:
            duration = time.time() - start_time
            logger.info(f"GeoServer is accessible at {geoserver_url}")
            record_test_result("geoserver_status_check", True, "GeoServer is accessible", duration)
            return True
        else:
            duration = time.time() - start_time
            error_msg = f"Status: {response.status_code}, Response: {response.text[:200]}"
            logger.error(f"GeoServer status check failed. {error_msg}")
            record_test_result("geoserver_status_check", False, error_msg, duration)
            return False
            
    except requests.exceptions.RequestException as e:
        duration = time.time() - start_time
        logger.error(f"Could not connect to GeoServer at {geoserver_url}: {e}")
        record_test_result("geoserver_status_check", False, f"Connection error: {e}", duration)
        return False